In [24]:
import pandas as pd
import country_converter as coco
import pycountry_convert as pc

In [16]:
df=pd.read_json('data/data.json')

display(df)

,id,name,origin_country,receiving_country,time,distance,date_sent,date_received,origin_region,receiving_region,origin_city,receiving_city,origin_iso,receiving_iso
0,UZ-1,UZ-1.jpg,Uzbekistan,Netherlands,80,4861,2018-05-02,2018-07-21,Tashkent Region,Provincie Utrecht,Yalanghoch,Zeist,UZ,NL
1,UZ-2,UZ-2.jpg,Uzbekistan,Latvia,27,3666,2020-03-03,2020-03-30,(general),Rīga,Tashkent,Riga,UZ,LV
2,UZ-3,UZ-3.jpg,Uzbekistan,Netherlands,18,4759,2013-02-23,2013-03-13,Tashkent Region,Provincie Drenthe,Yalanghoch,Coevorden,UZ,NL
3,UZ-4,UZ-4.jpg,Uzbekistan,Germany,15,4754,2024-01-03,2024-01-18,Tashkent,Nordrhein-Westfalen,Tashkent,Essen,UZ,DE
4,UZ-5,UZ-5.jpg,Uzbekistan,Malaysia,28,5294,2023-12-05,2024-01-03,Tashkent,Selangor,Tashkent,Kuala Kubu Baharu,UZ,MY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10595,US-46,US-46.jpg,U.S.A.,Türkiye,16,9849,2010-07-01,2010-07-16,Colorado,İstanbul,Greeley,Istanbul,US,TR
10596,US-47,US-47.jpg,U.S.A.,Ireland,25,5783,2023-12-06,2024-01-01,North Carolina,Leinster,Raleigh,Dublin,US,IE
10597,US-48,US-48.jpg,U.S.A.,U.S.A.,10,4022,2024-10-13,2024-10-22,Washington,Florida,Bothell,The Villages,US,US
10598,US-49,US-49.jpg,U.S.A.,Germany,21,8289,2022-03-24,2022-04-14,Texas,Lower Saxony,Austin,Marx,US,DE


# Mapping countries to continents

In [26]:
MANUAL_MAP = {
    'VA': 'EU',  # Vatican City -> Europe
    'TL': 'AS',  # Timor-Leste -> Asia
    'PS': 'AS',  # Palestine -> Asia
}

def get_continent(code):
    if pd.isna(code) or code == '':
        return "Unknown"

    # Checks manual overrides first
    if code in MANUAL_MAP:
        continent_code = MANUAL_MAP[code]
    else:
        try:
            # Tries standard conversion
            continent_code = pc.country_alpha2_to_continent_code(code)
        except KeyError:
            return "Unknown"

    # 3. Map to full name
    cn_map = {
        'AF': 'Africa', 'AS': 'Asia', 'EU': 'Europe',
        'NA': 'North America', 'OC': 'Oceania',
        'SA': 'South America', 'AN': 'Antarctica'
    }
    return cn_map.get(continent_code, "Unknown")

In [28]:
df['origin_continent'] = df['origin_iso'].apply(get_continent)
df['receiving_continent'] = df['receiving_iso'].apply(get_continent)

In [29]:
display(df['receiving_continent'].unique())

array(['Europe', 'Asia', 'North America', 'South America', 'Oceania',
       'Africa'], dtype=object)

In [30]:
CONTINENT_CENTERS = {
         "Asia": [87.6173, 34.0479],
         "Europe": [15.2551, 54.5260],
         "Africa": [18.5204, 8.7832],
         "North America": [-95.7129, 37.0902],
         "South America": [-58.3816, -34.6037],
         "Oceania": [133.7751, -25.2744]
}
df['origin_continent_coord'] = df['origin_continent'].map(CONTINENT_CENTERS)
df['receiving_continent_coord'] = df['receiving_continent'].map(CONTINENT_CENTERS)

In [31]:
df.to_json('data/data_with_geography.json',orient='records')